In [14]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25 --quiet
# !: for running the commands directly to the shell not as python code
# --quiet : for not showing the logs

In [15]:
!pip install nltk --quiet

In [16]:
import os
import json
import re
import time
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from rank_bm25 import BM25Okapi
from pydantic import Field
from typing import List

In [17]:
try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    STOP_WORDS = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [word for word in words if word not in STOP_WORDS]

    return " ".join(words)

In [18]:
corpus_filename = "news_corpus.json"
full_corpus = ""

with open(corpus_filename, "r", encoding="utf-8") as f:
    news_data = json.load(f)

for item in news_data:
    article_text = f"Title: {item.get('title', '')} \nContent: {item.get('text', '')} \nSource: {item.get('source', '')}"
    cleaned_article = clean_text(article_text)
    full_corpus += cleaned_article + "\n"

cleaned_txt_filename = "cleaned_corpus.txt"
with open(cleaned_txt_filename, "w", encoding="utf-8") as f:
    f.write(full_corpus)
print(f"Saved preprocessed corpus to '{cleaned_txt_filename}' with {len(news_data)} news items.")

Saved preprocessed corpus to 'cleaned_corpus.txt' with 10 news items.


In [19]:
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    PAGEINDEX_API_KEY = userdata.get("PAGEINDEX")
    print("API Keys loaded from Google Colab Secrets")
except:
    print("Running outside Colab")

API Keys loaded from Google Colab Secrets


In [20]:
# ==============================================================================
# CELL 7: Direct PageIndex Tree Navigation from Uploaded JSON File
# ==============================================================================
extracted_documents = []

def traverse_pageindex_tree(node, current_metadata=None):
    """
    Recursively walks through your uploaded hierarchical JSON tree topology
    (Root -> Titles -> Leaf Text Paragraphs). Extracts raw text strings,
    applies text cleanup routines, and packages them inside LangChain Documents.
    """
    if current_metadata is None:
        current_metadata = {}

    node_type = node.get("type", "unknown")
    node_title = node.get("title", "")

    # Track down the contextual structure of section headings dynamically
    if node_type in ["title", "heading"]:
        current_metadata["section_heading"] = node_title

    # Extract text from leaf nodes
    if "text" in node and node["text"]:
        clean_chunk = clean_text(node["text"])

        # Package text and section markers into a LangChain Document object
        doc = Document(
            page_content=clean_chunk,
            metadata={
                "source": "news_corpus.json",
                "section": current_metadata.get("section_heading", "General Summary")
            }
        )
        extracted_documents.append(doc)

    # Recurse through all child branches of the current block
    for child in node.get("children", []):
        traverse_pageindex_tree(child, current_metadata.copy())

# ------------------------------------------------------------------------------
# Dynamically load the tree directly from your uploaded file
# ------------------------------------------------------------------------------
corpus_filename = "news_corpus.json"

if not os.path.exists(corpus_filename):
    raise FileNotFoundError(f"Please upload '{corpus_filename}' using the left sidebar panel in Google Colab.")

with open(corpus_filename, "r", encoding="utf-8") as f:
    uploaded_tree_data = json.load(f)

# If your JSON root is a dictionary (like your image shows), pass it directly.
# If your file is a list of roots, loop over it.
if isinstance(uploaded_tree_data, list):
    for root_node in uploaded_tree_data:
        traverse_pageindex_tree(root_node)
else:
    traverse_pageindex_tree(uploaded_tree_data)

print(f"Extracted {len(extracted_documents)} hierarchical chunk nodes directly from your uploaded file!")

Extracted 10 hierarchical chunk nodes directly from your uploaded file!


In [21]:
class BM25Retriever(BaseRetriever):
    docs: List[Document] = Field(default_factory=list)
    bm25: BM25Okapi = Field(default=None)

    def _get_relevant_documents(self, query: str):
        query_tokens = query.lower().split()
        return self.bm25.get_top_n(
            query_tokens,
            self.docs,
            n=2
        )

In [22]:
tokenized_corpus = [
    doc.page_content.lower().split()
    for doc in extracted_documents
]

bm25_index = BM25Okapi(tokenized_corpus)

bm25_retriever = BM25Retriever(
    docs=extracted_documents,
    bm25=bm25_index
)

print(f"Initialized Retriever over {{len(extracted_documents)}} document blocks.")
print("Retriever Created")

Initialized Retriever over {len(extracted_documents)} document blocks.
Retriever Created


In [23]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

system_prompt = (
    "You are an assistant answering questions strictly based on the provided context.\n"
    "If the answer is not contained within the context, state 'Information not found.'\n\n"
    "Context:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [24]:
query_test = "What is the HTTP/2 Bomb vulnerability and what servers does it impact?"

retrieved_docs = bm25_retriever._get_relevant_documents(query_test)
context = "\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
Answer strictly from the supplied context.

Context:
{context}

Question:
{query_test}
"""

start_rag = time.time()
rag_response = llm.invoke(prompt)
end_rag = time.time()
rag_latency = end_rag - start_rag

In [25]:
start_naive = time.time()

naive_prompt = f"""
Answer strictly using the corpus.

Question:
{query_test}

Corpus:
{full_corpus}
"""

naive_response = llm.invoke(naive_prompt)
end_naive = time.time()
naive_latency = end_naive - start_naive

In [26]:
print("=" * 50)
print("BM25 RAG LATENCY")
print(rag_latency)
print("\nBM25 RAG ANSWER")
print(rag_response.content)
print("\n" + "=" * 50)
print("NAIVE LATENCY")
print(naive_latency)
print("\nNAIVE ANSWER")
print(naive_response.content)
print("=" * 50)

BM25 RAG LATENCY
4.87651252746582

BM25 RAG ANSWER
The HTTP/2 Bomb vulnerability, codenamed HTTP/2 Bomb, chains together two known techniques: a compression bomb and a slowloris-style hold bomb. It targets the HPACK HTTP/2 header compression scheme, where one byte on the wire becomes one full header allocation on the server, repeated thousands of times per request. A zero-byte flow control window keeps the server from freeing the HPACK dedicated header compression algorithm. This variant allocates per-entry bookkeeping memory around the decoded size limit, which never fires, resulting in almost nothing to decode, but still consuming server memory. A single client with a 100mbps connection could potentially render a vulnerable server inaccessible within seconds, consuming and holding 32GB of server memory in Apache HTTPD and Envoy within 20 seconds.

The HTTP/2 Bomb vulnerability impacts major web servers including Nginx, Apache HTTPD, Microsoft IIS, Envoy, Cloudflare, and Pingora.

NAI